# Signisa — Kaggle 1st-place recipe on OUR split (GPU, internet ON)

Diagnostic (memory.md, session 17): retrain hoyso48's asl-signs 1st-place recipe — their
preprocessing, model, and AWP / Lookahead / late-dropout training, ported in
`signisa.kaggle1st` and verified against their published TFLite — on the SAME 17/4 PopSign
signer split as every signisa run. Input: the **asl-signs** competition data only. Edit only CONFIG.

**Pre-registered** (merged 246-class top-1 at the best-val-loss checkpoint, their rule):
- **>= 75%** -> our recipe/pipeline is the gap. Next: experiment 2 (their model on our tensors).
- **< 60%** -> the split itself is hard; our ~50% is nearer its ceiling than thought.
- **60-75%** -> both contribute; run experiment 2.

signisa reference: 50.3% best val top-1 (ArcFace, 200 epochs, 246 classes).

In [ ]:
# CONFIG — the only cell to edit
EPOCHS = 300          # theirs; every epoch prints a projected total
BATCH_SIZE = 512      # their 8 TPU replicas x 64, run as 8 accumulated micro-batches of 64
LR = 4e-3
WORKERS = 4
TIME_BUDGET_H = 11.0  # stops (and still evaluates + saves) before Kaggle's 12 h kill

In [ ]:
# rm first: a re-run must never pip-install a stale clone
!rm -rf /kaggle/working/signisa-repo && git clone -q https://github.com/dayan-battulga/signisa.git /kaggle/working/signisa-repo
%pip install -q /kaggle/working/signisa-repo

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from signisa.config import Config
from signisa.eval import held_out_participants
from signisa.kaggle1st import (PointsDataset, RecipeConfig, asl_signs_root, logits_of,
                               points_cache, train_kaggle1st)

ROOT = asl_signs_root()  # any mount depth; on failure it lists what IS attached
print("asl-signs:", ROOT)
train = pd.read_csv(ROOT / "train.csv")
sign_index = json.loads((ROOT / "sign_to_prediction_index_map.json").read_text())
cache = points_cache([ROOT / p for p in train.path], train.sign.map(sign_index).to_numpy(), WORKERS)
print(f"{len(train)} sequences, {len(cache['frames'])} frames, mean T {cache['lengths'].mean():.1f}")

# the SAME seeded draw as every signisa run (signisa.eval.default_val_participants, PopSign part)
val_pids = held_out_participants(train.participant_id, Config().n_val_participants, Config().seed)
is_val = train.participant_id.isin(val_pids).to_numpy()
print("val participants:", val_pids, "| train", (~is_val).sum(), "| val", is_val.sum())

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "turn on the GPU accelerator"
cfg = RecipeConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, num_workers=WORKERS,
                   time_budget_h=TIME_BUDGET_H)
train_ds = PointsDataset(cache, np.flatnonzero(~is_val), augment=True)
val_ds = PointsDataset(cache, np.flatnonzero(is_val), augment=False)
model, history = train_kaggle1st(cfg, train_ds, val_ds, device)
torch.save(model.state_dict(), "/kaggle/working/kaggle1st_on_our_split.pt")
pd.DataFrame(history).rename_axis("epoch").to_csv("/kaggle/working/history.csv")

In [ ]:
REPO = "/kaggle/working/signisa-repo"
logits, labels = logits_of(model, val_ds, device)
probs, labels = logits.softmax(1).numpy(), labels.numpy()
# fold the 4 alias pairs (cat/kitty, ...) into signisa's 246 classes, as live_demo does
classes = json.loads(Path(f"{REPO}/data/meta/training_labels.json").read_text())["classes"]
group = np.empty(len(sign_index), dtype=int)
for ci, c in enumerate(classes):
    for member in c["members"]:
        group[sign_index[member]] = ci
merged_hit = (probs @ np.eye(len(classes))[group]).argmax(1) == group[labels]
merged, raw = merged_hit.mean(), (probs.argmax(1) == labels).mean()
by_signer = pd.Series(merged_hit).groupby(train.participant_id[is_val].to_numpy()).mean()
best = int(np.argmin(history["val_loss"]))
peak = int(np.argmax(history["val_top1"]))
verdict = ("PASS — our recipe/pipeline is the gap; next: experiment 2 (their model on our tensors)"
           if merged >= 0.75 else
           "KILL — the split itself is hard; our ~50% is nearer its ceiling than thought"
           if merged < 0.60 else
           "PARTIAL — recipe and split both contribute; run experiment 2")
report = f"""# Kaggle 1st-place recipe on signisa's 17/4 split

- Val signers {val_pids}: {is_val.sum()} sequences | train {(~is_val).sum()} sequences.
- Ran {len(history['val_loss'])}/{EPOCHS} epochs in {history['hours'][-1]:.1f} h.
- **Top-1 at best val loss (epoch {best + 1}): {merged:.1%} merged (246 classes), {raw:.1%} raw (250).**
- Peak per-epoch raw top-1 {history['val_top1'][peak]:.1%} (epoch {peak + 1}); final epoch {history['val_top1'][-1]:.1%}.
- signisa reference: 50.3% best val top-1 (ArcFace, 200 epochs, 246 classes).
- Per signer (merged): {', '.join(f'{p}: {v:.1%}' for p, v in by_signer.items())}
- **Verdict (pre-registered): {verdict}**
"""
Path("/kaggle/working/kaggle1st_report.md").write_text(report)
print(report)